<a href="https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

## Feature vector

The feature vector is built from the March 2026 partition of
`fact_content_daily_performance`.

Only historical metrics that are available before prediction are included.
Rows are filtered using `ga4_data_available IS TRUE` so GA4 metrics represent
real observations rather than zero-filled values.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    ga4_pageviews,
    ga4_sessions

FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')

WHERE month='2026-03'
AND ga4_data_available IS TRUE

LIMIT 20
""")

feature_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬────────────────────┬───────────────┬──────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │
│    date     │         varchar         │         varchar          │      int64      │   int64    │       double       │     int64     │    int64     │
├─────────────┼─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼────────────────────┼───────────────┼──────────────┤
│ 2026-03-01  │ client_65de48885f4ef01b │ content_09be8cc7fcb222af │               0 │          0 │               NULL │             1 │            1 │
│ 2026-03-01  │ client_65de48885f4ef01b │ content_851afac9fe13612e │               0 │          0 │               NULL │             1 │            1 │
│ 2026-03-01  │ client_65de48885f4ef01b │ content_cee6c6fc8c51af14 │               0 │  

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Feature notes

### 1. gsc_impressions
- **Meaning:** Number of times a content item appeared in Google Search results.
- **Missing values:** No special missing value handling was required.
- **Categorical:** No (numeric).
- **Available when?** Available before prediction because it is historical Search Console data.

### 2. gsc_clicks
- **Meaning:** Number of clicks received from Google Search.
- **Missing values:** No special missing value handling was required.
- **Categorical:** No (numeric).
- **Available when?** Available before prediction because it is historical Search Console data.

### 3. gsc_avg_position
- **Meaning:** Average ranking position of the content in Google Search.
- **Missing values:** NULL values indicate no ranking information was available for that day. These values are kept as NULL rather than replaced.
- **Categorical:** No (numeric).
- **Available when?** Available before prediction because it is calculated from historical search performance.

### 4. ga4_pageviews
- **Meaning:** Number of page views recorded in Google Analytics.
- **Missing values:** Only rows where `ga4_data_available IS TRUE` are included, so pageview values represent real observations.
- **Categorical:** No (numeric).
- **Available when?** Available before prediction because it is historical GA4 data.

### 5. ga4_sessions
- **Meaning:** Number of sessions recorded in Google Analytics.
- **Missing values:** Only rows where `ga4_data_available IS TRUE` are included.
- **Categorical:** No (numeric).
- **Available when?** Available before prediction because it is historical GA4 data.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_df.df().describe(include="all")

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
count,20,20,20,20.000000,20.000000,10.000000,20.000000,20.000000
unique,NaN,1,20,NaN,NaN,NaN,NaN,NaN
top,NaN,client_65de48885f4ef01b,content_09be8cc7fcb222af,NaN,NaN,NaN,NaN,NaN
freq,NaN,20,1,NaN,NaN,NaN,NaN,NaN
mean,2026-03-01 00:00:00,NaN,NaN,189.500000,0.150000,8.613831,1.150000,1.100000
min,2026-03-01 00:00:00,NaN,NaN,0.000000,0.000000,5.156425,1.000000,1.000000
25%,2026-03-01 00:00:00,NaN,NaN,0.000000,0.000000,5.419565,1.000000,1.000000
50%,2026-03-01 00:00:00,NaN,NaN,2.500000,0.000000,5.917276,1.000000,1.000000
75%,2026-03-01 00:00:00,NaN,NaN,39.250000,0.000000,7.405449,1.000000,1.000000
max,2026-03-01 00:00:00,NaN,NaN,3282.000000,1.000000,30.304348,2.000000,2.000000


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## Leakage hunt

Data leakage happens when a feature contains information that would not be available at the time a prediction is made. This can lead to unrealistically high model performance.

To demonstrate this, I intentionally create one label-derived feature (`leak_feature`) that is identical to the prediction target. I first train a simple model using only the legitimate features and record its accuracy. Then I add the leaked feature and retrain the model. The accuracy is expected to increase dramatically because the model has direct access to the answer. Finally, I remove the leaked feature to restore an honest evaluation.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Convert DuckDB relation to pandas
df = feature_df.df().copy()

# Remove rows with missing average position
df = df.dropna(subset=["gsc_avg_position"])

# Simple label: whether the page received at least one click
df["label"] = (df["gsc_clicks"] > 0).astype(int)

# ---------------- Honest model ----------------

X = df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
    ]
]

y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

honest_accuracy = accuracy_score(y_test, model.predict(X_test))
print("Honest accuracy:", round(honest_accuracy, 3))

# ---------------- Leakage model ----------------

X_leak = X.copy()
X_leak["leak_feature"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X_leak, y, test_size=0.3, random_state=42
)

model.fit(X_train, y_train)

leak_accuracy = accuracy_score(y_test, model.predict(X_test))
print("Accuracy with leakage:", round(leak_accuracy, 3))

Honest accuracy: 0.667
Accuracy with leakage: 1.0


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## What I excluded and why

The following fields were intentionally excluded from the feature vector:

- **client_hash_id** – Used only as an identifier for grouping and joins. It should not be used as a model feature.
- **content_hash_id** – A unique identifier for each content item. It contains no predictive information.
- **report_date** – Used only to define the observation window, not as a predictive feature.
- **month** – A partition column used for filtering data.
- **gsc_data_available** – Used only to determine whether Search Console data exists.
- **ga4_data_available** – Used only to filter valid GA4 observations.
- **Label-derived columns (for example, `leak_feature`)** – Excluded because they directly reveal the prediction target and cause data leakage, leading to unrealistically high model performance.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_features = {
    "client_hash_id": "Identifier only",
    "content_hash_id": "Identifier only",
    "report_date": "Observation date",
    "month": "Partition column",
    "gsc_data_available": "Availability flag",
    "ga4_data_available": "Availability flag",
    "leak_feature": "Label-derived (data leakage)"
}

print("Excluded Features:")
for feature, reason in excluded_features.items():
    print(f"- {feature}: {reason}")

Excluded Features:
- client_hash_id: Identifier only
- content_hash_id: Identifier only
- report_date: Observation date
- month: Partition column
- gsc_data_available: Availability flag
- ga4_data_available: Availability flag
- leak_feature: Label-derived (data leakage)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.